# IMDb NLP Processor - Actividad Práctica

## Unidad 2 - Clase 1: Text Mining y Embeddings Contextuales

Esta actividad trabaja el flujo completo de procesamiento de texto sobre el dataset IMDb de Kaggle, sin usar Transformers ni modelos preentrenados.

> 🧑‍🎓 **Versión para estudiantes.** Las celdas de código están vacías: complétalas en clase siguiendo las explicaciones.
> Cada celda trae **pistas** (funciones útiles) y los **nombres de variables** que deben quedar definidos, porque las celdas siguientes los usan.
> Ejecuta las celdas en orden. Las funciones de apoyo están en `src/` (revisa sus docstrings). La versión resuelta es `actividad.ipynb`.

## Objetivos de aprendizaje
- Leer y explorar un dataset real de reseñas.
- Limpiar texto de forma sistemática.
- Tokenizar y convertir a secuencias numéricas.
- Aplicar padding (pre/post) y justificar la elección.
- Visualizar una Embedding Layer y entender su salida.

In [ ]:
# ✅ Celda de configuración (ya resuelta): ejecútala antes de empezar.

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Resolver raiz del proyecto de forma robusta
def find_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / 'src').exists() and (folder / 'dataset').exists() and (folder / 'notebooks').exists():
            return folder
    raise FileNotFoundError('No se encontro la raiz de IMDb_NLP_Processor')

ROOT = find_root(Path.cwd())
SRC = ROOT / 'src'
IMAGES = ROOT / 'images'
IMAGES.mkdir(parents=True, exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

from utils import load_imdb_dataset, sample_dataset, dataset_status
from eda import (
    add_review_length,
    basic_report,
    sentiment_distribution,
    longest_reviews,
    shortest_reviews,
    plot_sentiment_distribution,
    plot_length_histogram,
)
from preprocessing import clean_dataframe, show_before_after
from tokenizer_utils import build_tokenizer, texts_to_sequences, top_n_words, apply_padding, suggest_max_len
from embedding_utils import build_embedding_model

print('Entorno listo. ROOT =', ROOT)

In [ ]:
# ✅ Celda de configuración (ya resuelta): ejecútala antes de empezar.

# --- BLOQUE OPCIONAL: ACTIVAR/CONFIGURAR GPU ---
# Si ya tienes tu entorno configurado, puedes dejarlo en False.
USE_GPU = False

if USE_GPU:
    import tensorflow as tf
    
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        # Evita que TensorFlow reserve toda la memoria de la GPU al inicio.
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU detectada(s): {len(gpus)}")
        for i, gpu in enumerate(gpus):
            print(f"  GPU {i}: {gpu.name}")
    else:
        print("No se detectó GPU. Se usará CPU.")
else:
    print("Bloque GPU opcional desactivado (USE_GPU=False).")

## Carga del dataset
El CSV esperado es `dataset/imdb_reviews.csv` con columnas `review` y `sentiment`.

Si aún no tienes el archivo real de Kaggle, el notebook creará una muestra mínima de demostración para no bloquear el flujo, pero debes reemplazarla para la entrega final.

In [ ]:
# ✍️ Ejercicio 1: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): load_imdb_dataset, DataFrame, dataset_status, head
# 📌 Al terminar deberían existir: csv_path, df


## Muestreo opcional para acelerar práctica
Puedes elegir entre usar todo el dataset o tomar una muestra (por ejemplo, 2,000 a 5,000 registros).

In [ ]:
# ✍️ Ejercicio 2: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): reset_index, sample_dataset, sample
# 📌 Al terminar deberían existir: USE_ALL_SAMPLES, N_SAMPLES


# Parte 1 - Exploración del Dataset (EDA)

In [ ]:
# ✍️ Ejercicio 3: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): basic_report
# 📌 Al terminar deberían existir: report


In [ ]:
# ✍️ Ejercicio 4: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): add_review_length, sentiment_distribution, mean
# 📌 Al terminar deberían existir: df_eda, dist


In [ ]:
# ✍️ Ejercicio 5: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): longest_reviews, shortest_reviews


In [ ]:
# ✍️ Ejercicio 6: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): plot_sentiment_distribution, plot_length_histogram


**Interpretación sugerida:**
- Verifica si las clases están balanceadas o no.
- Evalúa si hay reseñas extremadamente largas/cortas que puedan afectar el padding.
- Identifica si conviene truncar secuencias en un percentil de longitud.

# Parte 2 - Limpieza del Texto

In [ ]:
# ✍️ Ejercicio 7: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): clean_dataframe, show_before_after
# 📌 Al terminar deberían existir: df_clean


**¿Por qué estas transformaciones?**
- Minúsculas: reduce variación artificial (`Great` vs `great`).
- HTML/URLs: suelen ser ruido para sentimiento en este nivel.
- Números y signos: simplifican vocabulario inicial.
- Espacios repetidos: normaliza y evita tokens vacíos.

# Parte 3 - Tokenización

In [ ]:
# ✍️ Ejercicio 8: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): build_tokenizer, texts_to_sequences, top_n_words, head
# 📌 Al terminar deberían existir: NUM_WORDS, tokenizer, sequences


**Explicación técnica:** cada reseña se convierte en una lista de enteros según `word_index`. Ese diccionario asigna un índice único a cada palabra del vocabulario.

# Parte 4 - Padding

In [ ]:
# ✍️ Ejercicio 9: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): suggest_max_len, mean, apply_padding
# 📌 Al terminar deberían existir: p50, p90, MAX_LEN, recortadas, pad_pre, pad_post


**Estrategia seleccionada:** `post-padding`.

Justificación: en reseñas suele ser útil mantener el inicio del texto (contexto inicial), y agregar ceros al final facilita inspección y comparación de ejemplos en esta etapa formativa.

**Sobre `MAX_LEN`:** se elige con el percentil 90 de las longitudes en vez de un valor arbitrario: cubre la gran mayoría de reseñas sin llenar la matriz de ceros.

# Parte 5 - Introducción a Word Embeddings

In [ ]:
# ✍️ Ejercicio 10: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): build_embedding_model, summary, astype, lower
# 📌 Al terminar deberían existir: vocab_size, EMBEDDING_DIM, model, X, y


In [ ]:
# ✍️ Ejercicio 11: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): predict
# 📌 Al terminar deberían existir: sample_x, sample_out


**Explicación de capas:**
- `Embedding`: convierte IDs de palabras a vectores densos.
- `GlobalAveragePooling1D`: resume la secuencia en un único vector promedio.
- `Dense`: proyección lineal/no lineal mínima para obtener una salida escalar.
- `mask_zero=True`: el padding (índice 0) no participa en el promedio.
- `keras.Input(shape=(MAX_LEN,))` reemplaza a `input_length` (obsoleto en Keras 3).

# Parte 6 - Reflexión Técnica

Responde con argumentos técnicos:

1. ¿Por qué un modelo de IA no puede trabajar directamente con texto?
2. ¿Cuál es la función del Tokenizer?
3. ¿Qué representa el `word_index`?
4. ¿Por qué todas las secuencias deben tener la misma longitud?
5. ¿Qué ventaja ofrece una Embedding Layer frente a usar solo números enteros?
6. ¿Qué dificultades encontraste durante el procesamiento del dataset?

# Entrenamiento completo del modelo (enfoque de aprendizaje)
En esta sección entrenaremos un modelo más sólido para práctica real, evaluaremos su desempeño y haremos inferencia con textos escritos por el usuario.

> ⚠️ **Buena práctica:** dividimos train/test **antes** de ajustar el tokenizer, para que el vocabulario no "conozca" palabras del conjunto de prueba (fuga de información).

In [ ]:
# ✍️ Ejercicio 12: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): set_random_seed, train_test_split, build_tokenizer, apply_padding, unique, compute_class_weight, texts_to_sequences
# 📌 Al terminar deberían existir: RUN_FULL_TRAINING


### Bloque 2: definición y compilación del modelo
Se define una arquitectura más robusta y luego se compila con métricas útiles para seguimiento del aprendizaje.

In [ ]:
# ✍️ Ejercicio 13: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): Sequential, compile, summary, Input, Embedding, GlobalAveragePooling1D, Dense, Dropout


### Bloque 3: callbacks y entrenamiento
Entrenamos por más épocas con EarlyStopping y ReduceLROnPlateau para mejorar estabilidad.

La validación para EarlyStopping se toma del 10% de **train** (`validation_split`): el conjunto de test solo se usa una vez, al final.

In [ ]:
# ✍️ Ejercicio 14: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): fit, EarlyStopping, ReduceLROnPlateau


### Bloque 4: métricas y curvas de entrenamiento
Evaluamos métricas principales y visualizamos las curvas para analizar el comportamiento del modelo por época.

In [ ]:
# ✍️ Ejercicio 15: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): evaluate, items, figure, subplot, plot, title, xlabel, ylabel


### Bloque 5: matriz de confusión y reporte de clasificación
Se evalúa la calidad del modelo con métricas por clase y una matriz de confusión.

In [ ]:
# ✍️ Ejercicio 16: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): ravel, astype, confusion_matrix, ConfusionMatrixDisplay, plot, title, tight_layout, savefig


### Bloque 6: función de inferencia y ejemplos
Creamos una función reutilizable para predecir sentimiento sobre cualquier texto nuevo.

In [ ]:
# ✍️ Ejercicio 17: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): clean_text, texts_to_sequences, apply_padding, ravel, predict


### Bloque 7: predicción con tu propio texto
Escribe tu reseña en inglés en la variable `user_text` y vuelve a ejecutar la celda. Incluimos dos casos difíciles (negación y sarcasmo) para discutir las limitaciones del modelo.

In [ ]:
# ✍️ Ejercicio 18: escribe aquí tu código siguiendo la explicación de arriba.
# 💡 Pistas (funciones/métodos útiles): predict_sentiment


## Cierre pedagógico
Este repositorio queda listo para extenderse durante la Unidad 2 con embeddings avanzados, Transformers, BERT y fine-tuning en próximas clases.

**Limitación clave:** el promedio de embeddings ignora el orden de las palabras (`not bad` ≈ `bad`). Los **Transformers** con *self-attention* atacan justamente esto → próxima clase.